### LOADING AND CONFIG

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os 

DATA_PATH = r'D:\PJT2\CMAPSSData' 
PROCESSED_PATH = r'../../processed_data'
RUL_CAP = 125 
WINDOW_SIZE = 30 

# Columns
cols = ['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f's{i}' for i in range(1, 22)]

# Loading
print(f"Loading FD003 (Multi-Fault) from {DATA_PATH}...")
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train_FD003.txt'), sep='\s+', header=None, names=cols)
test_df = pd.read_csv(os.path.join(DATA_PATH, 'test_FD003.txt'), sep='\s+', header=None, names=cols)
print(f"Train Shape: {train_df.shape}")

Loading FD003 (Multi-Fault) from D:\PJT2\CMAPSSData...
Train Shape: (24720, 26)


### RUL calculation

In [2]:
def add_rul(df):
    max_cycle = df.groupby('unit_number')['time_in_cycles'].max().reset_index()
    max_cycle.columns = ['unit_number', 'max_cycle']
    df = df.merge(max_cycle, on='unit_number', how='left')
    df['RUL'] = df['max_cycle'] - df['time_in_cycles']
    df = df.drop(columns=['max_cycle'])
    df['RUL'] = df['RUL'].clip(upper=RUL_CAP)
    return df

train_df = add_rul(train_df)

### FEATURE SELECTION, SCALING AND SMOOTHING

In [3]:
# We only drop sensors that are STRICTLY constan
stds = train_df.describe().loc['std']
constant_features = stds[stds == 0].index.tolist()
print(f"Dropping Constant Features: {constant_features}")

# Keep settings? usually constant in FD003, but let's check.
drop_labels = constant_features + ['unit_number', 'time_in_cycles', 'RUL'] 
features = [c for c in train_df.columns if c not in drop_labels]
print(f"Training on {len(features)} features: {features}")

scaler = MinMaxScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

# Exponential Smoothing (Helps with the different fault types)
train_df[features] = train_df[features].ewm(alpha=0.1).mean()
test_df[features] = test_df[features].ewm(alpha=0.1).mean()

Dropping Constant Features: ['setting_3', 's1', 's18', 's19']
Training on 20 features: ['setting_1', 'setting_2', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's20', 's21']


### WINDOW GENERATION

In [4]:
def gen_sequence(id_df, seq_length, seq_cols):
    data_matrix = id_df[seq_cols].values
    num_elements = data_matrix.shape[0]
    for start, stop in zip(range(0, num_elements-seq_length), range(seq_length, num_elements)):
        yield data_matrix[start:stop, :]

def gen_labels(id_df, seq_length, label):
    data_matrix = id_df[label].values
    return data_matrix[seq_length:]

X_train, y_train = [], []
for unit_id in train_df['unit_number'].unique():
    unit_data = train_df[train_df['unit_number'] == unit_id]
    for seq in gen_sequence(unit_data, WINDOW_SIZE, features):
        X_train.append(seq)
    for label in gen_labels(unit_data, WINDOW_SIZE, 'RUL'):
        y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)
print(f"X_train Shape: {X_train.shape}")

# Test Gen
X_test, y_test = [], []
true_ruls = pd.read_csv(os.path.join(DATA_PATH, 'RUL_FD003.txt'), header=None, names=['RUL'])

for unit_id in test_df['unit_number'].unique():
    unit_data = test_df[test_df['unit_number'] == unit_id]
    if len(unit_data) >= WINDOW_SIZE:
        seq = unit_data[features].values[-WINDOW_SIZE:]
        X_test.append(seq)
        y_test.append(true_ruls.iloc[unit_id-1].values[0])

X_test = np.array(X_test)
y_test = np.array(y_test)
print(f"X_test Shape: {X_test.shape}")

X_train Shape: (21720, 30, 20)
X_test Shape: (100, 30, 20)


### SAVING THE DATA

In [5]:
if not os.path.exists(PROCESSED_PATH):
    os.makedirs(PROCESSED_PATH)

np.save(os.path.join(PROCESSED_PATH, 'X_train_FD003.npy'), X_train)
np.save(os.path.join(PROCESSED_PATH, 'y_train_FD003.npy'), y_train)
np.save(os.path.join(PROCESSED_PATH, 'X_test_FD003.npy'), X_test)
np.save(os.path.join(PROCESSED_PATH, 'y_test_FD003.npy'), y_test)